In [1]:
"""
QLD Drought Data Consolidation Script
--------------------------------------
Merges yearly NetCDF files (2000-2025) for SPI, SPEI, and Protracted Drought
into three consolidated, QLD-clipped, grid-aligned files ready for Zenodo.

HOW TO RUN (Google Colab, free):
1. Open https://colab.research.google.com/, new notebook.
2. First cell:
       from google.colab import drive
       drive.mount('/content/drive')
3. Edit BASE_DIR below to match your Drive folder path.
4. Paste this whole script into a cell and run it.
5. Read the printed output carefully -- especially the "SPEI alignment
   check" line and any WARNING lines about file counts.
6. Check processed/manifest.csv when done.

Expected input structure under BASE_DIR:
    SPI/                  <- 26 yearly .nc files
    SPEI/                 <- 26 yearly .nc files (full-Australia extent)
    Protracted Drought/   <- 26 yearly .nc files

If anything errors out or the alignment check shows a lot of NaNs, send
me the printed output and I'll adjust the script.
"""

import os
import glob
import xarray as xr
import pandas as pd

# ---- CONFIG: edit this one path ----
BASE_DIR = r"D:\Project Works\Drought_QLD\Data\Final DB for Project"
# --------------------------------------

SPI_DIR = os.path.join(BASE_DIR, "SPI")
SPEI_DIR = os.path.join(BASE_DIR, "SPEI")
PD_DIR = os.path.join(BASE_DIR, "Protracted Drought")
OUTPUT_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# QLD bounding box, generous buffer -- SPEI gets coarsely clipped to this
# first, then precisely snapped onto SPI's exact grid below.
QLD_LAT_MIN, QLD_LAT_MAX = -29.3, -9.9
QLD_LON_MIN, QLD_LON_MAX = 137.9, 153.6


def load_and_concat(folder, label):
    """Load all yearly .nc files in a folder and concatenate along time."""
    files = sorted(glob.glob(os.path.join(folder, "*.nc")))
    print(f"\n[{label}] {folder}: found {len(files)} files")
    if len(files) != 26:
        print(f"  WARNING: expected 26 files (2000-2025), found {len(files)}")

    datasets = []
    for f in files:
        ds = xr.open_dataset(f)
        dims_present = [d for d in ("time", "lat", "lon") if d in ds.dims]
        ds = ds.transpose(*dims_present, ...)
        datasets.append(ds)

    combined = xr.concat(datasets, dim="time", data_vars="minimal", compat="equals")
    combined = combined.sortby("time")
    for ds in datasets:
        ds.close()
    print(f"  Combined: {combined.sizes.get('time')} time steps, "
          f"{combined.sizes.get('lat')}x{combined.sizes.get('lon')} grid, "
          f"variables: {list(combined.data_vars)}")
    return combined


# ---- 1. SPI (already QLD-clipped) ----
spi = load_and_concat(SPI_DIR, "SPI")
spi_out = os.path.join(OUTPUT_DIR, "QLD_SPI_2000_2025.nc")
spi.to_netcdf(spi_out)
print(f"Saved: {spi_out}")

# Keep SPI's exact lat/lon grid as the reference for aligning SPEI
qld_lat_ref = spi["lat"]
qld_lon_ref = spi["lon"]

# ---- 2. SPEI (full-Australia extent -- clip + align to SPI's grid) ----
spei = load_and_concat(SPEI_DIR, "SPEI")

spei = spei.sortby("lat")
spei = spei.sel(
    lat=slice(QLD_LAT_MIN, QLD_LAT_MAX),
    lon=slice(QLD_LON_MIN, QLD_LON_MAX),
)

spei_aligned = spei.reindex(lat=qld_lat_ref, lon=qld_lon_ref, method="nearest", tolerance=0.01)

first_var = list(spei_aligned.data_vars)[0]
n_nan = spei_aligned[first_var].isel(time=0).isnull().sum().item()
n_total = spei_aligned[first_var].isel(time=0).size
print(f"SPEI alignment check ({first_var}, first time step): "
      f"{n_nan}/{n_total} NaN cells after snapping to SPI grid "
      f"({100*n_nan/n_total:.1f}%) -- should be low; high % means grids don't overlap well")

spei_out = os.path.join(OUTPUT_DIR, "QLD_SPEI_2000_2025.nc")
spei_aligned.to_netcdf(spei_out)
print(f"Saved: {spei_out}")

# ---- 3. Protracted Drought (already QLD-clipped, keep spi_01 duplicate) ----
pd_ds = load_and_concat(PD_DIR, "Protracted Drought")

# bioregion_id has no time dimension in source files, so concat should
# leave it untouched -- this is just a safety check in case it picked one up.
if "bioregion_id" in pd_ds.data_vars and "time" in pd_ds["bioregion_id"].dims:
    pd_ds["bioregion_id"] = pd_ds["bioregion_id"].isel(time=0, drop=True)

pd_out = os.path.join(OUTPUT_DIR, "QLD_protracted_drought_label_2000_2025.nc")
pd_ds.to_netcdf(pd_out)
print(f"Saved: {pd_out}")

# ---- 4. Manifest ----
manifest = []
for name, ds, path in [
    ("SPI", spi, spi_out),
    ("SPEI", spei_aligned, spei_out),
    ("Protracted Drought", pd_ds, pd_out),
]:
    manifest.append({
        "index": name,
        "file": os.path.basename(path),
        "variables": ", ".join(ds.data_vars),
        "time_steps": ds.sizes.get("time"),
        "time_start": str(pd.to_datetime(ds["time"].values.min())) if "time" in ds.dims else "n/a",
        "time_end": str(pd.to_datetime(ds["time"].values.max())) if "time" in ds.dims else "n/a",
        "n_lat": ds.sizes.get("lat"),
        "n_lon": ds.sizes.get("lon"),
        "file_size_mb": round(os.path.getsize(path) / 1e6, 1),
    })

manifest_df = pd.DataFrame(manifest)
manifest_path = os.path.join(OUTPUT_DIR, "manifest.csv")
manifest_df.to_csv(manifest_path, index=False)
print("\n=== Manifest ===")
print(manifest_df.to_string(index=False))
print(f"\nSaved manifest to: {manifest_path}")
print("\nDone. Check the SPEI alignment check line above before proceeding.")


[SPI] D:\Project Works\Drought_QLD\Data\Final DB for Project\SPI: found 26 files
  Combined: 312 time steps, 382x311 grid, variables: ['spi_01', 'spi_03', 'spi_06', 'spi_12']
Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\QLD_SPI_2000_2025.nc

[SPEI] D:\Project Works\Drought_QLD\Data\Final DB for Project\SPEI: found 26 files
  Combined: 312 time steps, 681x841 grid, variables: ['spei_gamma_01', 'spei_gamma_12', 'spei_gamma_03', 'spei_gamma_06']
SPEI alignment check (spei_gamma_01, first time step): 51657/118802 NaN cells after snapping to SPI grid (43.5%) -- should be low; high % means grids don't overlap well
Saved: D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\QLD_SPEI_2000_2025.nc

[Protracted Drought] D:\Project Works\Drought_QLD\Data\Final DB for Project\Protracted Drought: found 26 files
  Combined: 312 time steps, 382x311 grid, variables: ['spi_01', 'protracted_drought', 'bioregion_id']
Saved: D:\Project Works\Drought_QLD\Data\Final D

In [2]:
import xarray as xr

spi = xr.open_dataset(r"D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\QLD_SPI_2000_2025.nc")
spei = xr.open_dataset(r"D:\Project Works\Drought_QLD\Data\Final DB for Project\processed\QLD_SPEI_2000_2025.nc")

spi_valid = spi["spi_01"].isel(time=0).notnull()
spei_valid = spei["spei_gamma_01"].isel(time=0).notnull()

both_valid = (spi_valid & spei_valid).sum().item()
spi_only = (spi_valid & ~spei_valid).sum().item()      # SPI has data, SPEI alignment failed here -- bad sign
spei_only = (~spi_valid & spei_valid).sum().item()
both_nan = (~spi_valid & ~spei_valid).sum().item()      # likely shared ocean mask -- fine

total = spi_valid.size
print(f"Total cells: {total}")
print(f"Both valid (good overlap): {both_valid} ({100*both_valid/total:.1f}%)")
print(f"SPI valid but SPEI NaN (likely misalignment): {spi_only} ({100*spi_only/total:.1f}%)")
print(f"SPEI valid but SPI NaN: {spei_only} ({100*spei_only/total:.1f}%)")
print(f"Both NaN (likely shared ocean/no-data mask): {both_nan} ({100*both_nan/total:.1f}%)")

Total cells: 118802
Both valid (good overlap): 60941 (51.3%)
SPI valid but SPEI NaN (likely misalignment): 0 (0.0%)
SPEI valid but SPI NaN: 6204 (5.2%)
Both NaN (likely shared ocean/no-data mask): 51657 (43.5%)
